In [1]:
import numpy as np
import pandas as pd
import fastf1 as f1

In [2]:
circuits = pd.read_csv('data/circuits.csv')
constructor_results = pd.read_csv('data/constructor_results.csv')
constructor_standings = pd.read_csv('data/constructor_standings.csv')
constructors = pd.read_csv('data/constructors.csv')
driver_standings = pd.read_csv('data/driver_standings.csv')
drivers = pd.read_csv('data/drivers.csv')
lap_times = pd.read_csv('data/lap_times.csv')
pit_stops = pd.read_csv('data/pit_stops.csv')
qualifying = pd.read_csv('data/qualifying.csv')
races = pd.read_csv('data/races.csv')
results = pd.read_csv('data/results.csv')
sprint_results = pd.read_csv('data/sprint_results.csv')
status = pd.read_csv('data/status.csv')
hungary = f1.get_session(2024, 13, "Race")
belgium = f1.get_session(2024, 14, "Race")

req         WARNING 	DEFAULT CACHE ENABLED! (291.12 MB) /Users/jestonlewis/Library/Caches/fastf1


In [3]:
hungary.load()
belgium.load()

core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.4.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '44', '16', '1', '55', '11', '63', '22', '18', '14', '3', '27', '23', '20', '77', '2', '31', '24', '10']
core           INFO 	Loading data for Belgian Grand Prix - R

In [4]:
final = pd.merge(results, races, on="raceId")
final = pd.merge(final, circuits, on="circuitId")
final = pd.merge(final, drivers, on="driverId")

In [5]:
final.drop(columns=["resultId", "number_x", "positionText", "positionOrder", "points", "laps", "time_x", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "round", "name_x", "url_x", "fp1_date", "fp1_time", "fp2_date", "fp2_time", "fp3_date", "fp3_time", "quali_date", "quali_time", "sprint_date", "sprint_time", "name_y", "location", "country", "lat", "lng", "url_y", "number_y", "code", "forename", "surname", "dob", "nationality", "url", "alt"], inplace=True)

final = pd.merge(final, constructors, on="constructorId")

In [6]:
final.drop(columns=["name", "nationality", "url"], inplace=True)

In [7]:
final.rename(columns={"time_y":"time"}, inplace=True)

In [8]:
final.drop(final[final.year < 2010].index, inplace=True)

In [9]:
races = [belgium, hungary]

In [10]:
for race in races:
    race_results = race.results
    race_results.drop(columns=["DriverNumber", "BroadcastName", "Abbreviation", "TeamColor", "FirstName", "LastName", "FullName", "HeadshotUrl", "CountryCode", "ClassifiedPosition", "Q1", "Q2", "Q3", "Time", "Status"], inplace=True)
    race_results = pd.merge(race_results, drivers, left_on="DriverId", right_on="driverRef")
    race_results["year"] = race.session_info["StartDate"].date().year
    race_results["date"] = race.session_info["StartDate"].date().strftime("%Y-%m-%d")
    race_results["time"] = race.session_info["StartDate"].time().strftime("%H:%M")
    if race.session_info["Meeting"]["Circuit"]["ShortName"] == "Spa-Francorchamps":
        # TODO come up with better way to add circuit names
        race_results["circuitRef"] = "spa"
        race_results["raceId"] = 1134
    else:
        race_results["circuitRef"] = "hungaroring"
        race_results["raceId"] = 1133
    race_results = pd.merge(race_results, circuits, on="circuitRef")
    race_results = pd.merge(race_results, constructors, left_on="TeamId", right_on="constructorRef")
    race_results.drop(columns=["DriverId", "TeamName", "TeamId", "Points", "number", "code", "country", "lat", "lng", "alt", "url_y", "name_y", "nationality_y", "url", "forename", "surname", "dob", "nationality_x", "url_x", "name_x", "location"], inplace=True)
    race_results.rename(columns={"Position":"position", "GridPosition":"grid"}, inplace=True)
    final = pd.concat([final, race_results], axis=0)

In [11]:
final.replace(to_replace="\\N", value=np.nan, inplace=True)

In [12]:
final.drop(columns=["driverId", "constructorId", "circuitId"], inplace=True)

In [37]:
def update_names(local_table, column):
    refs = local_table[column]
    real = np.array([])
    for ref in refs:
        if ref.find("_") != -1:
            ref = ref.replace("_", " ")
        ref = ref.title()
        real = np.append(real, ref)

    return real

In [38]:
final["circuitRef"] = update_names(final, "circuitRef")
final["constructorRef"] = update_names(final, "constructorRef")
final["driverRef"] = update_names(final, "driverRef")

In [39]:
final["circuit_code"] = final["circuitRef"].astype("category").cat.codes
final["driver_code"] = final["driverRef"].astype("category").cat.codes
final["constructor_code"] = final["constructorRef"].astype("category").cat.codes

In [40]:
final["position"] = final["position"].astype(float)

In [41]:
final = final[final.position <= 20.0]

In [42]:
final["pos_delta"] = final["grid"] - final["position"]

In [44]:
final

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta
20320,337,3.0,1.0,2010,2010-03-14,12:00:00,Bahrain,Alonso,Ferrari,2,3,5,2.0
20321,337,2.0,2.0,2010,2010-03-14,12:00:00,Bahrain,Massa,Ferrari,2,41,5,0.0
20322,337,4.0,3.0,2010,2010-03-14,12:00:00,Bahrain,Hamilton,Mclaren,2,23,13,1.0
20323,337,1.0,4.0,2010,2010-03-14,12:00:00,Bahrain,Vettel,Red Bull,2,72,17,-3.0
20324,337,5.0,5.0,2010,2010-03-14,12:00:00,Bahrain,Rosberg,Mercedes,2,59,14,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,1133,12.0,16.0,2024,2024-07-21,15:00,Hungaroring,Bottas,Sauber,7,7,19,-4.0
16,1133,14.0,17.0,2024,2024-07-21,15:00,Hungaroring,Sargeant,Williams,7,63,22,-3.0
17,1133,19.0,18.0,2024,2024-07-21,15:00,Hungaroring,Ocon,Alpine,7,49,2,1.0
18,1133,18.0,19.0,2024,2024-07-21,15:00,Hungaroring,Zhou,Sauber,7,76,19,-1.0


In [45]:
def rolling_finish_avg(group, cols, new_cols):
    group = group.sort_values("raceId")
    rolling_stats = group[cols].rolling(3, closed="left").mean()
    group[new_cols] = rolling_stats
    return group

In [46]:
cols = ["grid", "position", "pos_delta"]
new_cols = [f"{c}_rolling" for c in cols]

In [47]:
final_rolling = final.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")

/var/folders/hk/ffpx1y0s2cn133v15g3yks080000gn/T/ipykernel_1531/536465426.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rolling = final.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")


In [48]:
final_rolling = final_rolling.sort_values(["raceId", "position"])

In [49]:
final_rolling.index = range(final_rolling.shape[0])

In [50]:
final_rolling.to_csv("data/final_rolling.csv", index=False)

In [51]:
single = final_rolling.loc[final_rolling["raceId"] == 1134]

In [52]:
single["constructorRef"].values

array(['Mercedes', 'Mclaren', 'Ferrari', 'Red Bull', 'Mclaren', 'Ferrari',
       'Red Bull', 'Aston Martin', 'Alpine', 'Rb', 'Aston Martin',
       'Williams', 'Alpine', 'Haas', 'Sauber', 'Rb', 'Williams', 'Haas',
       'Sauber', 'Mercedes'], dtype=object)

In [53]:
dutch_gp = pd.DataFrame(
    {
        "raceId": [1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135, 1135],
        "grid": [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
        "position": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
        "pos_delta": [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
        "driver_code": [343, 617, 461, 519, 580, 699, 607,  20, 583, 670, 756,  14, 290,
       425, 102, 798, 705, 379, 855, 693],
        "driverRef": ['hamilton', 'piastri', 'leclerc', 'max_verstappen', 'norris',
       'sainz', 'perez', 'alonso', 'ocon', 'ricciardo', 'stroll', 'albon',
       'gasly', 'kevin_magnussen', 'bottas', 'tsunoda', 'sargeant',
       'hulkenberg', 'zhou', 'russell'],
        "constructor_code": [136, 131,  69, 165, 131,  69, 165,  11,   5, 162,  11, 208,   5,
        79, 168, 162, 208,  79, 168, 136],
        "constructorRef": ['mercedes', 'mclaren', 'ferrari', 'red_bull', 'mclaren', 'ferrari',
       'red_bull', 'aston_martin', 'alpine', 'rb', 'aston_martin',
       'williams', 'alpine', 'haas', 'sauber', 'rb', 'williams', 'haas',
       'sauber', 'mercedes'],
        "circuit_code": [39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39],
        "circuitRef": ["zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort", "zandvoort","zandvoort"]
    }
)

In [54]:
with_single = pd.concat([final, dutch_gp])

In [55]:
with_single

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta
20320,337,3.0,1.0,2010.0,2010-03-14,12:00:00,Bahrain,Alonso,Ferrari,2,3,5,2.0
20321,337,2.0,2.0,2010.0,2010-03-14,12:00:00,Bahrain,Massa,Ferrari,2,41,5,0.0
20322,337,4.0,3.0,2010.0,2010-03-14,12:00:00,Bahrain,Hamilton,Mclaren,2,23,13,1.0
20323,337,1.0,4.0,2010.0,2010-03-14,12:00:00,Bahrain,Vettel,Red Bull,2,72,17,-3.0
20324,337,5.0,5.0,2010.0,2010-03-14,12:00:00,Bahrain,Rosberg,Mercedes,2,59,14,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,1135,NaN,16.0,NaN,NaN,NaN,zandvoort,tsunoda,rb,39,798,162,NaN
16,1135,NaN,17.0,NaN,NaN,NaN,zandvoort,sargeant,williams,39,705,208,NaN
17,1135,NaN,18.0,NaN,NaN,NaN,zandvoort,hulkenberg,haas,39,379,79,NaN
18,1135,NaN,19.0,NaN,NaN,NaN,zandvoort,zhou,sauber,39,855,168,NaN


In [56]:
dutch_rolling = with_single.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")

/var/folders/hk/ffpx1y0s2cn133v15g3yks080000gn/T/ipykernel_1531/64606415.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dutch_rolling = with_single.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")


In [57]:
dutch_rolling = dutch_rolling.loc[dutch_rolling["raceId"] == 1135]

In [58]:
dutch_rolling

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta,grid_rolling,position_rolling,pos_delta_rolling
11,1135,NaN,12.0,NaN,NaN,NaN,zandvoort,albon,williams,39,14,208,NaN,NaN,NaN,NaN
7,1135,NaN,8.0,NaN,NaN,NaN,zandvoort,alonso,aston_martin,39,20,11,NaN,NaN,NaN,NaN
14,1135,NaN,15.0,NaN,NaN,NaN,zandvoort,bottas,sauber,39,102,168,NaN,NaN,NaN,NaN
12,1135,NaN,13.0,NaN,NaN,NaN,zandvoort,gasly,alpine,39,290,5,NaN,NaN,NaN,NaN
0,1135,NaN,1.0,NaN,NaN,NaN,zandvoort,hamilton,mercedes,39,343,136,NaN,NaN,NaN,NaN
17,1135,NaN,18.0,NaN,NaN,NaN,zandvoort,hulkenberg,haas,39,379,79,NaN,NaN,NaN,NaN
13,1135,NaN,14.0,NaN,NaN,NaN,zandvoort,kevin_magnussen,haas,39,425,79,NaN,NaN,NaN,NaN
2,1135,NaN,3.0,NaN,NaN,NaN,zandvoort,leclerc,ferrari,39,461,69,NaN,NaN,NaN,NaN
3,1135,NaN,4.0,NaN,NaN,NaN,zandvoort,max_verstappen,red_bull,39,519,165,NaN,NaN,NaN,NaN
4,1135,NaN,5.0,NaN,NaN,NaN,zandvoort,norris,mclaren,39,580,131,NaN,NaN,NaN,NaN


In [59]:
dutch_rolling.to_csv("data/dutch_rolling.csv", index=False)